# Part B — WGAN and WGAN-GP
**Dataset:** CIFAR-10 (32×32 RGB). Pixels normalized to [-1, 1] for the generator's tanh output; FID is computed on images rescaled back to [0, 1].

Implements:
1. **WGAN** with weight clipping (Arjovsky et al., 2017).
2. **WGAN-GP** with gradient penalty (Gulrajani et al., 2017).

Reports training curves (critic loss / Wasserstein distance estimate), 100-image sample grids, and FID scores.

## Runtime
- **Colab:** `Runtime → Change runtime type → GPU (T4 or better)`, then Run-All. Deps auto-install in the setup cell below.
- **WILP lab GPU:** same code; skip the Drive-mount cell.


## 0. Colab / environment setup

In [ ]:
import sys, subprocess, importlib
IN_COLAB = 'google.colab' in sys.modules

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg, mod in [('torch-fidelity', 'torch_fidelity')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        _pip(pkg)

# ---- Mount Google Drive and point at the assignment folder ----
# Drive layout (per the assignment):
#   MyDrive / Semester 2 / UDL - ELEC / Assignment2 / data /
#       batches.meta, data_batch_1..5, test_batch, readme.html
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = '/content/drive/MyDrive/Semester 2/UDL - ELEC/Assignment2'
else:
    BASE_DIR = '.'

DATA_DIR = f'{BASE_DIR}/data'
OUT_DIR  = f'{BASE_DIR}/outputs_part_b'

from pathlib import Path
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('IN_COLAB:', IN_COLAB)
print('BASE_DIR:', BASE_DIR)
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR :', OUT_DIR)

expected = ['batches.meta', 'data_batch_1', 'data_batch_2', 'data_batch_3',
            'data_batch_4', 'data_batch_5', 'test_batch']
missing = [f for f in expected if not Path(DATA_DIR, f).exists()]
if missing:
    print('MISSING CIFAR-10 files in', DATA_DIR, ':', missing)
else:
    print('All CIFAR-10 pickle files found.')

import torch
print('CUDA available:', torch.cuda.is_available(),
      '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('WARNING: no GPU detected. On Colab: Runtime -> Change runtime type -> GPU.')


In [ ]:
import os, math, time, random, json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, ' Torch:', torch.__version__)


In [ ]:
class CFG:
    data_root       = DATA_DIR
    batch_size      = 128
    num_workers     = 2
    z_dim           = 128
    g_ch            = 64
    d_ch            = 64
    # WGAN clip
    wgan_epochs     = 60
    wgan_lr         = 5e-5
    wgan_clip       = 0.01
    n_critic        = 5
    # WGAN-GP
    wgangp_epochs   = 60
    wgangp_lr       = 1e-4
    gp_lambda       = 10.0
    beta1, beta2    = 0.0, 0.9
    # FID
    fid_n_samples   = 5000
    out_dir         = OUT_DIR

Path(CFG.out_dir).mkdir(parents=True, exist_ok=True)


## 1. CIFAR-10 data loader (raw Python pickle format)
Loads the CIFAR-10 batches straight from Drive using the format documented at
`https://www.cs.toronto.edu/~kriz/cifar.html`:

```python
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')
```

Each row of `b'data'` is 3072 uint8 laid out as **red[1024] || green[1024] || blue[1024]** — we reshape to `(3, 32, 32)`.
Assignment says pixels in `[0, 1]`; the WGAN generator uses `tanh` so we then remap train tensors to `[-1, 1]`. Test tensors used as FID reference stay in `[0, 1]`.


In [ ]:
import pickle
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

def unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

class CIFAR10Pickle(Dataset):
    # Raw CIFAR-10 python pickle format (cs.toronto.edu/~kriz/cifar.html).
    def __init__(self, root, train=True, transform=None):
        root = Path(root)
        files = [root / f'data_batch_{i}' for i in range(1, 6)] if train else [root / 'test_batch']
        data_chunks, label_chunks = [], []
        for fp in files:
            d = unpickle(str(fp))
            data_chunks.append(d[b'data'])
            label_chunks.extend(d[b'labels'])
        self.data   = np.concatenate(data_chunks, axis=0).reshape(-1, 3, 32, 32).astype(np.uint8)
        self.labels = np.asarray(label_chunks, dtype=np.int64)
        self.transform = transform
        meta_fp = root / 'batches.meta'
        if meta_fp.exists():
            meta = unpickle(str(meta_fp))
            self.classes = [s.decode('utf-8') for s in meta[b'label_names']]
        else:
            self.classes = [str(i) for i in range(10)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = np.transpose(self.data[idx], (1, 2, 0))   # HWC uint8
        img = Image.fromarray(img)
        if self.transform is not None:
            img = self.transform(img)
        return img, int(self.labels[idx])

tfm_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),   # [0,1] -> [-1,1]
])
tfm_test = transforms.Compose([transforms.ToTensor()])         # [0,1] for FID reference

train_ds = CIFAR10Pickle(CFG.data_root, train=True,  transform=tfm_train)
test_ds  = CIFAR10Pickle(CFG.data_root, train=False, transform=tfm_test)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, pin_memory=True)

def denorm(x):
    # [-1,1] -> [0,1]
    return (x.clamp(-1, 1) + 1) / 2

print(f'Train: {len(train_ds)}  Test: {len(test_ds)}  Classes: {train_ds.classes}')
_x, _y = next(iter(train_loader))
print(f'Train batch shape {_x.shape} range [{_x.min():.2f}, {_x.max():.2f}] (expected [-1,1])')
_x, _y = next(iter(test_loader))
print(f'Test  batch shape {_x.shape} range [{_x.min():.2f}, {_x.max():.2f}] (expected [ 0, 1])')


## 2. FID via torch-fidelity

In [ ]:
import subprocess, tempfile

def _dump_images(tensor, folder):
    Path(folder).mkdir(parents=True, exist_ok=True)
    for i, t in enumerate(tensor):
        vutils.save_image(t.clamp(0,1).cpu(), Path(folder) / f'{i:05d}.png')

def compute_fid(real_imgs, fake_imgs, cache_tag='fid'):
    try:
        import torch_fidelity
    except ImportError:
        subprocess.check_call(['pip', 'install', 'torch-fidelity', '--quiet'])
        import torch_fidelity
    with tempfile.TemporaryDirectory() as td:
        rd, fd = Path(td)/'real', Path(td)/'fake'
        _dump_images(real_imgs, rd)
        _dump_images(fake_imgs, fd)
        metrics = torch_fidelity.calculate_metrics(
            input1=str(rd), input2=str(fd), fid=True, verbose=False,
            cuda=torch.cuda.is_available()
        )
    return metrics['frechet_inception_distance']

def gather_real_test_images(n=None):
    n = n or CFG.fid_n_samples
    xs = []
    for x, _ in test_loader:
        xs.append(x)
        if sum(t.size(0) for t in xs) >= n: break
    return torch.cat(xs, 0)[:n]

def save_grid(imgs, path, nrow=10):
    grid = vutils.make_grid(imgs.clamp(0,1).cpu(), nrow=nrow, padding=1)
    vutils.save_image(grid, path)
    return grid


## 3. Generator and Critic (shared architectures)
DCGAN-style generator; critic uses LayerNorm (no BN, per WGAN-GP recommendation) so the same critic class is safe for both WGAN and WGAN-GP training.


In [ ]:
class Generator(nn.Module):
    def __init__(self, z=CFG.z_dim, ch=CFG.g_ch):
        super().__init__()
        self.net = nn.Sequential(
            # z -> 4x4
            nn.ConvTranspose2d(z, ch*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ch*8), nn.ReLU(True),
            # 4 -> 8
            nn.ConvTranspose2d(ch*8, ch*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ch*4), nn.ReLU(True),
            # 8 -> 16
            nn.ConvTranspose2d(ch*4, ch*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ch*2), nn.ReLU(True),
            # 16 -> 32
            nn.ConvTranspose2d(ch*2, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z.view(z.size(0), z.size(1), 1, 1))

class Critic(nn.Module):
    # Uses LayerNorm (recommended for WGAN-GP), also fine for WGAN.
    def __init__(self, ch=CFG.d_ch):
        super().__init__()
        def block(cin, cout, sz):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 4, 2, 1, bias=False),
                nn.LayerNorm([cout, sz, sz]),
                nn.LeakyReLU(0.2, inplace=True),
            )
        self.net = nn.Sequential(
            # 32 -> 16
            nn.Conv2d(3, ch, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, inplace=True),
            block(ch,   ch*2, 8),   # 16 -> 8
            block(ch*2, ch*4, 4),   # 8  -> 4
            block(ch*4, ch*8, 2),   # 4  -> 2
            nn.Conv2d(ch*8, 1, 2, 1, 0, bias=False),
        )
    def forward(self, x):
        return self.net(x).view(-1)

def sample_z(n):
    return torch.randn(n, CFG.z_dim, device=DEVICE)


## 4. WGAN (weight clipping)
- RMSProp optimizer (per original paper).
- Critic weights clipped to [-c, c] after each critic update.
- Generator updated every `n_critic` critic steps.


In [ ]:
def train_wgan():
    G = Generator().to(DEVICE)
    D = Critic().to(DEVICE)
    optG = torch.optim.RMSprop(G.parameters(), lr=CFG.wgan_lr)
    optD = torch.optim.RMSprop(D.parameters(), lr=CFG.wgan_lr)

    hist = {'epoch': [], 'd_loss': [], 'g_loss': [], 'w_dist': []}
    for ep in range(1, CFG.wgan_epochs + 1):
        d_running = g_running = w_running = 0.0
        d_n = g_n = 0
        G.train(); D.train()
        for step, (x, _) in enumerate(train_loader):
            x = x.to(DEVICE)
            # --- Critic step ---
            z = sample_z(x.size(0))
            with torch.no_grad():
                fake = G(z)
            d_real = D(x).mean()
            d_fake = D(fake).mean()
            d_loss = -(d_real - d_fake)     # maximize d_real - d_fake
            optD.zero_grad(); d_loss.backward(); optD.step()
            for p in D.parameters():
                p.data.clamp_(-CFG.wgan_clip, CFG.wgan_clip)
            d_running += d_loss.item(); w_running += (d_real - d_fake).item(); d_n += 1

            # --- Generator step every n_critic ---
            if step % CFG.n_critic == 0:
                z = sample_z(x.size(0))
                g_loss = -D(G(z)).mean()
                optG.zero_grad(); g_loss.backward(); optG.step()
                g_running += g_loss.item(); g_n += 1

        hist['epoch'].append(ep)
        hist['d_loss'].append(d_running / max(1, d_n))
        hist['g_loss'].append(g_running / max(1, g_n))
        hist['w_dist'].append(w_running / max(1, d_n))
        print(f"WGAN ep {ep:02d}  D {hist['d_loss'][-1]:+.4f}  G {hist['g_loss'][-1]:+.4f}  W~{hist['w_dist'][-1]:+.4f}")
    return G, D, hist

G_wgan, D_wgan, hist_wgan = train_wgan()
torch.save(G_wgan.state_dict(), Path(CFG.out_dir) / 'G_wgan.pt')
torch.save(D_wgan.state_dict(), Path(CFG.out_dir) / 'D_wgan.pt')
with open(Path(CFG.out_dir) / 'wgan_hist.json', 'w') as f:
    json.dump(hist_wgan, f, indent=2)


## 5. WGAN-GP (gradient penalty)
- Adam(0.0, 0.9) per Gulrajani et al.
- Gradient penalty enforces ‖∇D(x̂)‖₂ ≈ 1 on samples interpolated between real and fake.
- No weight clipping.


In [ ]:
def gradient_penalty(D, real, fake):
    bsz = real.size(0)
    eps = torch.rand(bsz, 1, 1, 1, device=DEVICE)
    hat = eps * real + (1 - eps) * fake
    hat.requires_grad_(True)
    d_hat = D(hat)
    grads = torch.autograd.grad(d_hat, hat,
                                 grad_outputs=torch.ones_like(d_hat),
                                 create_graph=True, retain_graph=True)[0]
    grads = grads.view(bsz, -1)
    gp = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp

def train_wgan_gp():
    G = Generator().to(DEVICE)
    D = Critic().to(DEVICE)
    optG = torch.optim.Adam(G.parameters(), lr=CFG.wgangp_lr, betas=(CFG.beta1, CFG.beta2))
    optD = torch.optim.Adam(D.parameters(), lr=CFG.wgangp_lr, betas=(CFG.beta1, CFG.beta2))

    hist = {'epoch': [], 'd_loss': [], 'g_loss': [], 'gp': [], 'w_dist': []}
    for ep in range(1, CFG.wgangp_epochs + 1):
        d_run = g_run = gp_run = w_run = 0.0
        d_n = g_n = 0
        G.train(); D.train()
        for step, (x, _) in enumerate(train_loader):
            x = x.to(DEVICE)
            # --- Critic step ---
            z = sample_z(x.size(0))
            with torch.no_grad():
                fake = G(z)
            d_real = D(x).mean()
            d_fake = D(fake).mean()
            gp = gradient_penalty(D, x, fake)
            d_loss = -(d_real - d_fake) + CFG.gp_lambda * gp
            optD.zero_grad(); d_loss.backward(); optD.step()
            d_run += d_loss.item(); w_run += (d_real - d_fake).item(); gp_run += gp.item(); d_n += 1

            # --- Generator step every n_critic ---
            if step % CFG.n_critic == 0:
                z = sample_z(x.size(0))
                g_loss = -D(G(z)).mean()
                optG.zero_grad(); g_loss.backward(); optG.step()
                g_run += g_loss.item(); g_n += 1

        hist['epoch'].append(ep)
        hist['d_loss'].append(d_run / d_n)
        hist['g_loss'].append(g_run / max(1, g_n))
        hist['gp'].append(gp_run / d_n)
        hist['w_dist'].append(w_run / d_n)
        print(f"WGAN-GP ep {ep:02d}  D {hist['d_loss'][-1]:+.4f}  G {hist['g_loss'][-1]:+.4f}  GP {hist['gp'][-1]:.4f}  W~{hist['w_dist'][-1]:+.4f}")
    return G, D, hist

G_gp, D_gp, hist_gp = train_wgan_gp()
torch.save(G_gp.state_dict(), Path(CFG.out_dir) / 'G_wgangp.pt')
torch.save(D_gp.state_dict(), Path(CFG.out_dir) / 'D_wgangp.pt')
with open(Path(CFG.out_dir) / 'wgangp_hist.json', 'w') as f:
    json.dump(hist_gp, f, indent=2)


## 6. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist_wgan['epoch'], hist_wgan['w_dist'], label='WGAN',    marker='o', markersize=3)
ax[0].plot(hist_gp['epoch'],  hist_gp['w_dist'],  label='WGAN-GP', marker='o', markersize=3)
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Estimated Wasserstein distance')
ax[0].set_title('Critic Wasserstein estimate'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(hist_wgan['epoch'], hist_wgan['g_loss'], label='WGAN G',    marker='o', markersize=3)
ax[1].plot(hist_gp['epoch'],   hist_gp['g_loss'],   label='WGAN-GP G', marker='o', markersize=3)
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Generator loss')
ax[1].set_title('Generator loss'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(Path(CFG.out_dir) / 'wgan_curves.png', dpi=150)
plt.show()


## 7. Random samples and FID

In [ ]:
@torch.no_grad()
def sample_gan(G, n):
    G.eval()
    z = sample_z(n)
    return denorm(G(z))       # [-1,1] -> [0,1]

# 100-image grids (Part C uses these)
grid_wgan   = sample_gan(G_wgan,  100).cpu()
grid_wgangp = sample_gan(G_gp,    100).cpu()
save_grid(grid_wgan,   Path(CFG.out_dir) / 'grid100_wgan.png',   nrow=10)
save_grid(grid_wgangp, Path(CFG.out_dir) / 'grid100_wgangp.png', nrow=10)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(vutils.make_grid(grid_wgan.clamp(0,1),   nrow=10, padding=1).permute(1,2,0));   ax[0].set_title('WGAN samples');    ax[0].axis('off')
ax[1].imshow(vutils.make_grid(grid_wgangp.clamp(0,1), nrow=10, padding=1).permute(1,2,0)); ax[1].set_title('WGAN-GP samples'); ax[1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
real_imgs = gather_real_test_images(CFG.fid_n_samples)

def collect_fake(G, n):
    fakes = []
    for i in range(0, n, CFG.batch_size):
        m = min(CFG.batch_size, n - i)
        fakes.append(sample_gan(G, m).cpu())
    return torch.cat(fakes, 0)

fake_wgan   = collect_fake(G_wgan,  CFG.fid_n_samples)
fake_wgangp = collect_fake(G_gp,    CFG.fid_n_samples)

fid_wgan   = compute_fid(real_imgs, fake_wgan,   cache_tag='wgan')
fid_wgangp = compute_fid(real_imgs, fake_wgangp, cache_tag='wgangp')
print(f'WGAN    FID = {fid_wgan:.2f}')
print(f'WGAN-GP FID = {fid_wgangp:.2f}')

with open(Path(CFG.out_dir) / 'wgan_fid.json', 'w') as f:
    json.dump({'WGAN': fid_wgan, 'WGAN-GP': fid_wgangp}, f, indent=2)


## 8. Part B summary
- **WGAN** with weight clipping tends to be more sensitive to the clipping value and can produce lower-fidelity samples (higher FID).
- **WGAN-GP** with gradient penalty produces sharper, more diverse samples and typically achieves noticeably lower FID at the same compute budget on CIFAR-10.
- Gradient penalty is more compute-heavy per critic step (double backward), so WGAN-GP epochs are somewhat slower.
